# Classification et validation des modèles

**Cartomize · Version 1.0**

Durée indicative : 40 minutes.

## Objectifs

Comparer K-moyennes, forêt aléatoire et Extra Trees ; exporter les modèles, les classes et la confiance.

## Apport de Cartomize

Cartomize coordonne l’échantillonnage, l’apprentissage, l’application par blocs et les rapports associés aux sorties.

Données : paysage pédagogique généré avec une graine fixe. Les cartes et mesures ne décrivent aucun site réel.

## Préparation

Installer l’environnement décrit dans le [guide de démarrage](../README.md), puis exécuter les cellules dans l’ordre. Chaque exécution utilise un nouveau dossier de résultats.

In [1]:
from pathlib import Path
import sys, json
import numpy as np
import pandas as pd
import geopandas as gpd
import rasterio
import matplotlib.pyplot as plt
from IPython.display import display, Image
root = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p/'course.py').is_file() or (p/'tutorials/course.py').is_file())
root = root if (root/'course.py').is_file() else root/'tutorials'
if str(root) not in sys.path:
    sys.path.insert(0, str(root))
import cartomize as cm
from course import demo, run_directory, show_raster, show_rgb, gallery, CRS, CLASSES, ROLES

In [2]:
out = run_directory("04_classification")
d = demo(out/"data")

## 1. Examiner les références

Les points d’apprentissage et de validation sont distincts, mais proviennent du même paysage simulé. Les scores mesurent la réussite sur cet exercice, pas une précision thématique sur le terrain.

In [3]:
training = cm.read_file(d['training'])
validation = cm.read_file(d['validation'])
display(training.groupby('libelle').size().rename('apprentissage'))
display(validation.groupby('libelle').size().rename('validation'))

libelle
Eau simulée           60
Sol simulé            60
Végétation simulée    60
Name: apprentissage, dtype: int64

libelle
Eau simulée           30
Sol simulé            30
Végétation simulée    30
Name: validation, dtype: int64

## 2. Classification non supervisée

Les groupes spectraux n’ont pas de signification thématique automatique. Un groupe ne doit pas être renommé « forêt » sans références.

In [4]:
clusters = cm.cluster_raster(d['image'], out/'kmeans', clusters=3, random_state=42, sample_limit=3000, block_size=64)

## 3. Apprentissage supervisé et transfert du modèle

In [5]:
models={}; predictions={}; reports=[]
for algorithm in ('random_forest','extra_trees'):
    model=cm.fit_classifier(d['image'], training, 'classe', label_column='libelle',
        validation=validation, algorithm=algorithm, n_estimators=40, random_state=42)
    model.save(out/(algorithm+'_model'))
    restored=cm.load_classifier(out/(algorithm+'_model'))
    predictions[algorithm]=cm.classify_landcover(d['image'], None, out/algorithm, model=restored, block_size=64)
    models[algorithm]=restored
    reports.append({'algorithme':algorithm, **{k:model.report['validation'][k] for k in ('pixels','accuracy','balanced_accuracy','kappa')}})
display(pd.DataFrame(reports))

,algorithme,pixels,accuracy,balanced_accuracy,kappa
0,random_forest,90,1.0,1.0,1.0
1,extra_trees,90,1.0,1.0,1.0


In [6]:
fig,axes=plt.subplots(1,3,figsize=(12,4),layout='constrained')
for ax,(name,path) in zip(axes,[('K-moyennes',clusters),*predictions.items()]):
    with rasterio.open(path) as src: ax.imshow(src.read(1,masked=True),cmap='viridis',interpolation='nearest',vmin=1,vmax=3)
    ax.set_title(name);ax.set_axis_off()
fig.suptitle('Classification d’un paysage simulé')
gallery(fig,'04_classification');plt.show()

## 4. Contrôler les classes et interpréter la confiance

La confiance est la probabilité maximale du modèle. Elle n’est pas une probabilité certifiée d’exactitude de chaque pixel. Examiner aussi la matrice de confusion, les classes minoritaires et la représentativité spatiale des références.

In [7]:
report=models['random_forest'].report
confusion=pd.DataFrame(report['validation']['confusion_matrix'],index=report['validation']['codes'],columns=report['validation']['codes'])
confusion.index.name='Référence';confusion.columns.name='Prédiction'
display(confusion)
with rasterio.open(predictions['random_forest']) as src:
    assert set(np.unique(src.read(1,masked=True).compressed())) <= {1,2,3}
assert report['validation_method']=='independent_features'

Prédiction,1,2,3
Référence,,,
1,30,0,0
2,0,30,0
3,0,0,30


## Exercice

Augmenter le bruit spectral ou réduire le nombre de points d’apprentissage. Quelle mesure permet de vérifier que toutes les classes restent correctement représentées ?

<details>
<summary>Éléments de correction</summary>

Examiner les rappels par classe et la balanced accuracy, en complément de l’exactitude globale. Sur des données réelles, réserver des références indépendantes, de préférence séparées spatialement.

</details>

## Reproduction

Les paramètres, la graine et les chemins sont explicites dans les cellules. Le répertoire `out` contient les produits de cette exécution. Adapter les sources, le système de coordonnées et les paramètres avant de transférer la méthode à une étude.